 # Analysis Overview Flow PBMC Frequency Analysis
This notebook contains the following analyses:

## 1. All NDMM subjects vs Healthy Donors

- Comparative analyses between **ALL** NDMM participants in this study and **healthy donor** controls.
- Evaluation of differences across relevant clinical and/or molecular features.
- Statistical testing and visualization to assess group-level variation.

## 2. Paired Analyses Across Clinical Time Points (All NDMM)

- Longitudinal, paired analyses within **ALL** NDMM participants.
- Comparison of matched samples collected at different clinical time points.
- Within-subject statistical testing to evaluate temporal changes over the course of the study.
- Visualization of trajectories and paired differences.


In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(purrr)
  library(ggplot2)
  library(ggpubr)
  library(rstatix)
  library(forcats)
  library(tidyr)
  library(ggrepel)
  library(rlang)
  library(scales)
  
})
options(repr.plot.width = 11, repr.plot.height = 6)

## Load Metadata

In [2]:
### Load Metadata 
meta = readxl::read_xlsx('../data/flow/ndmm-rrmm-metadata-2025.xlsx',sheet = 1)
meta$subject.subjectGuid <- meta$Subject

### delete first 3 rows as 
### these are reference ranges 
### and descriptors for each column 
meta = meta[-c(1:3),]

### Refactor all 'BRI' subjects as healthy
meta[meta$Cohort %in% c('BR1','BR2'),]$Cohort <- 'Healthy'
table(meta$Cohort)


    FH1 Healthy 
    197      34 

## Load Compositional Data

### Combine all files and summarize experiments

In [3]:
### Load combined files 
mmFiles = fread('../data/flow/output/aggregated_flowdata.csv')

In [4]:
# ### Join metadata with data files
# mmFiles <- dplyr::left_join(mmFiles, distinct(meta[,c('subject.subjectGuid')]),
#                       by='subject.subjectGuid')

### Keep only pre-transplant visits for this analysis 
mmFiles <- mmFiles[label.visitDetails %in% c('PreTx','PI2C','EI','Healthy')]
mmFiles$label.visitDetails <- factor(mmFiles$label.visitDetails,
                                    levels=c('PreTx','PI2C','EI','Healthy'))

# Analysis

## Compare Healthy vs. Cancer patients 

In [5]:
### Get cell pop names
uniqueCells = unique(mmFiles$aifi_label_l2)

### Create function to pull healthy data and data
### From cancer patients and compare their compositions 
test_timePoint <- function(time='PreTx'){
    
    ## repeat for each visit 
    timepoint_DF = mmFiles[label.visitDetails %in% c(time, "Healthy")]
    res <- lapply(uniqueCells,
                  function(x){
                      tmp=timepoint_DF[aifi_label_l2==x]
                      res = data.frame(
                          NDMM = median(tmp[Cohort=='FH1']$pseudo_alc, na.rm=T),
                          Healthy =  median(tmp[Cohort=='Healthy']$pseudo_alc, na.rm=T),
                          panel= unique(tmp$panel),
                          Time=time,
                          Cell=x,
                          Pval=wilcox.test(tmp[Cohort=='Healthy']$pseudo_alc,
                                           tmp[Cohort=='FH1']$pseudo_alc)$p.value)
                      }
                  )
    ### combine all results across cell populations  
    res = rbindlist(res)

    ### correct for multiple comparisons 
    res$FDR = p.adjust(res$Pval, method='fdr')

    ### calculate effect sizes 
    res$Log2FC = log2(res$NDMM) - log2(res$Healthy)
    return(res)    
}

### run all comparisons 
flow_comparisons <- rbindlist(lapply(c('PreTx','PI2C','EI'),
       function(x) test_timePoint(x)))

Warning message in wilcox.test.default(tmp[Cohort == "Healthy"]$pseudo_alc, tmp[Cohort == :
“cannot compute exact p-value with ties”
Warning message in wilcox.test.default(tmp[Cohort == "Healthy"]$pseudo_alc, tmp[Cohort == :
“cannot compute exact p-value with ties”


In [6]:
write.csv(flow_comparisons, '../data/flow/output/ndmm-vs-healthy-comparisons.csv')